In [6]:
import sys
import pandas as pd
import json
import os
import matplotlib.pyplot as plt
from jiwer import RemovePunctuation

!{sys.executable} -m pip install chat_gpt_asr

In [7]:
from chat_gpt_asr.alignment import *

In [8]:
sequences = [
    #REF, ASR, LLM
    ("short one here", "shoe order one here", "shorts one her"),
    ("I eat salami pizza salami", "I meat a salami pizza salami", "I eat a big pizza"),
]

In [9]:
ref,asr,llm = sequences[0]
align3(ref, asr, llm)

(['short', '', 'one', 'here'],
 ['shoe', 'order', 'one', 'here'],
 ['shorts', '', 'one', 'her'])

In [10]:
for ref, asr, llm in sequences:
    print_alignment(align3(ref, asr, llm))
    print()

        0      1    2     3
0   short         one  here
1    shoe  order  one  here
2  shorts         one   her

   0     1  2       3      4       5
0  I   eat     salami  pizza  salami
1  I  meat  a  salami  pizza  salami
2  I   eat  a     big  pizza        



In [11]:
#sequences_1 = [
    # REF, ASR, LLM
    #("Ummm I go to school", "Um I go to school", "I go to school"),
#]


In [12]:
path = "/home/mnaderi/Documents/thesis/chat-gpt-asr/results/results-dev-set/results_noisy/results_tiny/results_sentence_confidence_tiny/results_GPT-3.5-Turbo_tiny/gpt-3.5-turbo-0125/results_without_sentence_confidence_tiny/corrected_transcriptions_sentence_confidence_tiny.json"

with open(path, "r") as f:
    data = json.load(f)
    
    transcriptions = [RemovePunctuation()(d["asr_transcription"]["text"]).lower().strip() for d in data]
    reference_transcriptions = [RemovePunctuation()(d["reference_transcription"]).lower().strip() for d in data]
    corrected_transcriptions = [RemovePunctuation()(d["corrected_asr_transcription"]).lower().strip() for d in data]

In [13]:
for i, (ref, asr, llm) in enumerate(zip(reference_transcriptions, transcriptions, corrected_transcriptions)):
    if i > 2000 and i<2010:
        print_alignment(align3(ref, asr, llm))
        print()

    0        1   2       3    4       5   6    7     8         9   ...    37  \
0  one  morning  as   kanti  was  seated  in  his  boat  cleaning  ...  edge   
1  one  morning  as  gandhi  was  seated  in  his  boat  cleaning  ...  edge   
2  one  morning  as  gandhi  was  seated  in  his  boat  cleaning  ...  edge   

     38   39     40    41         42       43  44   45       46  
0  with  two  white        ducklings  clasped  to  her   breast  
1  with  two  white  tuck      links  clasped  to  her  pressed  
2  with  two  white        ducklings  clasped  to  her   breast  

[3 rows x 47 columns]

    0     1    2    3      4     5    6      7    8        9     10         11
0  the  girl  put  the  birds  into  the  water  and  watched  them  anxiously
1  the  girl  put  the  birds  into  the  water  and    watch  them  anxiously
2  the  girl  put  the  birds  into  the  water  and  watched  them  anxiously

        0      1      2      3    4    5   6    7     8         9    10  \

In [14]:
def identify_edit_type(ref, asr, llm):
    edit_types = []
    edits = {"A":[],"B":[],"C":[],"D":[]}
    rr, aa, ll = align3(ref, asr, llm)
    if len(rr) == len(aa) == len(ll):
        for r,a,l in zip(rr,aa,ll):
            if a == l == r:
                edit_types.append("C")  # left it correct
                edits["C"].append((a,l,r))
            elif a != l and l == r:
                edit_types.append("A")  # improve it
                edits["A"].append((a,l,r))
            elif a != r and l != r:
                edit_types.append("D")  # left it incorrect
                edits["D"].append((a,l,r))
            elif a == r and l != r:
                edit_types.append("B")  # introducing an error
                edits["B"].append((a,l,r))
    else:
        raise Exception
    return edit_types, edits

In [15]:
data = []
for i, (ref, asr, llm) in enumerate(zip(reference_transcriptions, transcriptions, corrected_transcriptions)):
    if i > 10:
        break
    edit_types, edits = identify_edit_type(ref, asr, llm)
    word_counts = {'A': 0, 'B': 0, 'C': 0, 'D': 0}
    for edit_type in edit_types:
        word_counts[edit_type] += 1
    
    # Create a DataFrame to store word counts by edit type
    data.append(word_counts)
    # df_edit_types = pd.DataFrame(word_counts.items(), columns=['Type', 'Count'])
    
    # Display the DataFrame
    # print("asr: {} len:{} \nllm: {} len: {} \nref: {} len: {}".format(asr, len(asr), llm, len(llm), ref, len(ref)))
    # print('edits: ', edits)
    # print(i, df_edit_types , "\n")

In [16]:
pd.DataFrame(data)

,A,B,C,D
0,1,0,21,0
1,1,0,9,0
2,0,0,13,0
3,2,0,25,0
4,0,0,10,1
5,0,2,4,3
6,0,0,6,1
7,3,0,26,5
8,0,0,7,0
9,1,0,11,2


In [ ]:
#df_edit_types

In [ ]:
#edits